# WWW::HorizonsEphemerisSystem basic usage

Anton Antonov  
April 2026

---

## Introduction

This notebook has examples for the basic usage of the sub `horizons-ephemeris-data` provided by the package ["WWW::HorizonsEphemerisSystem"](https://raku.land/zef:antononcube/WWW::HorizonsEphemerisSystem).

---

## Setup

In [1]:
use WWW::HorizonsEphemerisSystem;
use Data::Summarizers;
use Data::TypeSystem;

----

## Basic usage

Cartesian position of Mars with default parameters.

In [2]:
my $mars-default = horizons-ephemeris-data(
    'state',
    '499',
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Nov-27 14:31:08.0000, Date => 2461372.104953703, Vx => -21.39487770895718, Vy => -7.359345020622282, Vz => 0.3702789226879726, X => -94398908.2746674, Y => 222525059.7335831, Z => 7003303.986831605}]

Cartesian position of Mars for a specific date.

In [3]:
my $date = "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')}" with DateTime.now(:12hours);

my $mars-at-date = horizons-ephemeris-data(
    'state',
    ['499', {
        dates => $date,
    }],
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Nov-27 00:00:00.0000, Date => 2461371.500000000, Vx => -21.44049181153535, Vy => -7.250543072849926, Vz => 0.3736780107277102, X => -93279446.55129047, Y => 222906875.8378729, Z => 6983861.389652953}]

Azimuth/Elevation of the Moon from Moscow (lat, lon) for next 12 hours.

In [4]:
my @dates = do with DateTime.now { ($_, $_.later(:12hours), $_.later(minutes => 12*60 + 10))};
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });
say (:@dates);

my $tycho-from-moscow = horizons-ephemeris-data(
    'observer',
    {
        target => '301',
        center => (55.7505, 37.6175),
        :@dates,
    },
    <azimuth elevation>,
    :modifier<data>,
);

dates => [2026-11-27 09:31:10 2026-11-27 21:31:10 2026-11-27 21:41:10]


[{Azimuth => 325.479577, Elevation => -2.865972, ObservationDateUT => 2026-11-27T09:31:09.999955Z} {Azimuth => 113.351725, Elevation => 43.190488, ObservationDateUT => 2026-11-27T21:31:09.999955Z} {Azimuth => 115.905496, Elevation => 44.412096, ObservationDateUT => 2026-11-27T21:41:10.000003Z}]

Here is a range of dates from now to 4 hours from now with step 30 min:

In [5]:
my $start = DateTime.now;
my $end = $start.later(:4hours);
my $current = $start;
my @dates;

while $current <= $end {
    @dates.push($current);
    $current = $current.later(:30minutes);
}

#@dates = @dates».Str.map({ $_.substr(^19).subst('T', ' ') });
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });


deduce-type(@dates);

Vector(Atom((Str)), 9)

Orbital elements of Saturn moon Pandora for a specific list of date-times.

In [6]:
my $pandora-elements = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '500@699',
        :@dates
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

In [7]:
#%html
$pandora-elements.values
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Nov-27 13:31:10.0000,2461372.063310185,143595.2843578782,169.4379210490451,0.008705350388399343,28.02219045055302,358.9026401449539,0.006569876093435679,54795.55396177039,2.461372065243391E+06,141116.7661979897,160.1775795048299,142356.0252779339,358.8833254643411
A.D. 2026-Nov-27 11:01:10.0000,2461371.959143518,143414.1724910528,169.4372409733938,0.007452668069615175,28.02246352345791,330.5391631480617,0.006570067467142528,54793.95787035536,2.461372011042819E+06,141292.3492869864,128.8725065970998,142353.2608890196,330.1157054527598
A.D. 2026-Nov-27 10:31:10.0000,2461371.938310185,143340.8150643801,169.4371155683549,0.006944578248102486,28.02246688515265,324.8949358459665,0.006570138226669534,54793.3677466155,2.461372000151900E+06,141363.662542057,122.5833876116573,142352.2388032185,324.4340274955507
A.D. 2026-Nov-27 12:01:10.0000,2461372.000810185,143525.6849067545,169.4375513232298,0.008224219758610056,28.02240595987482,341.870883259418,0.006569952144509473,54794.91967089182,2.461372032747669E+06,141184.1685048557,141.4078727587863,142354.9267058051,341.5747477028974
A.D. 2026-Nov-27 13:01:10.0000,2461372.042476852,143585.6985616703,169.4378301648526,0.008639083051435856,28.02227429310307,353.2240316127603,0.006569886542332874,54795.46681367333,2.461372054413978E+06,141126.0501190976,153.9222803877481,142355.8743403839,353.1059625513518
A.D. 2026-Nov-27 11:31:10.0000,2461371.979976852,143476.0607384977,169.4373914888027,0.007880989599767881,28.02244436929653,336.2003267229836,0.006570004598315155,54794.48219751935,2.461372021903639E+06,141232.2772875472,135.1448218572239,142354.1690130224,335.8325807762737
A.D. 2026-Nov-27 12:31:10.0000,2461372.021643518,143562.3939938989,169.4377029430577,0.008478007020561708,28.02234817375446,347.5463128081929,0.006569912246343136,54795.25243284319,2.461372043582913E+06,141148.6120855264,147.6661698253328,142355.5030397127,347.3346174582649
A.D. 2026-Nov-27 09:31:10.0000,2461371.896643518,143163.4784347912,169.436979017091,0.005714505141867514,28.02245242583477,313.7147928675738,0.006570291942259379,54792.08582567245,2.461371978178467E+06,141536.5586092134,109.8972313429316,142350.0185220023,313.2391440982865
A.D. 2026-Nov-27 10:01:10.0000,2461371.917476852,143256.9083741799,169.4370265951987,0.00636289195459006,28.02246060725939,319.2800102843945,0.006570214061406387,54792.73531050527,2.461371989209093E+06,141445.3784836786,116.2650991383084,142351.1434289292,318.8014720878507


Using date range spec (the third element is automatically recognized to be a time step):

In [8]:
my $start = DateTime.now;
my $end = $start.later(:12hours);
my @date-range = $start, $end, "10 m";
@date-range .= map({ $_ ~~ DateTime ?? "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" !! $_ });
say (:@date-range);

my $pandora-elements-range = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '699',
        dates => @date-range
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

date-range => [2026-11-27 09:31:11 2026-11-27 21:31:11 10 m]


Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

Show a sample:

In [9]:
#%html
$pandora-elements-range.values
==> { .pick(6).sort(*<Date>)}()
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements-range.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Nov-27 13:01:11.0000,2461372.042488426,143585.7077164603,169.4378302262717,0.008639146325828819,28.02227424861564,353.2271862403152,0.006569886532222536,54795.46689799757,2.461372054419994E+06,141126.0412564011,153.9257556898034,142355.8744864307,353.1091710087902
A.D. 2026-Nov-27 13:21:11.0000,2461372.056377315,143593.6293363686,169.4378953290016,0.008693902878760817,28.0222189915549,357.0128826370153,0.006569877836603114,54795.53942301828,2.461372061639683E+06,141118.3708582588,158.0959909029003,142356.0000973137,356.9603974027934
A.D. 2026-Nov-27 14:21:11.0000,2461372.098043981,143580.4943550892,169.4379800656121,0.00860323901353281,28.02205021740009,8.369999912959504,0.006569893515588875,54795.40865400653,2.461372083298689E+06,141131.0528644468,170.6067782367109,142355.773609768,8.515048869708979
A.D. 2026-Nov-27 18:21:11.0000,2461372.264710648,143031.5779288476,169.4380879830408,0.004798226132156916,28.02188860839435,53.27158722525419,0.00657039310533638,54791.24220248148,2.461372170870099E+06,141665.5367932385,221.172926539259,142348.557361043,53.71385121550131
A.D. 2026-Nov-27 18:31:11.0000,2461372.271655092,142995.342252394,169.4381331051397,0.004546082238903394,28.02188604572396,55.05247976327332,0.006570416770357156,54791.04485793997,2.461372174677762E+06,141701.0888634192,223.3696098409031,142348.2155579066,55.48087483083101
A.D. 2026-Nov-27 18:41:11.0000,2461372.278599537,142958.3918306151,169.4381808872315,0.004288834509263913,28.02188174216488,56.81021269244324,0.006570439635630311,54790.85418390953,2.461372178526232E+06,141737.3787849575,225.5894348140039,142347.8853077863,57.22270723955531


Here is summary for some of the columns:

In [10]:
my @field-names = <Date Eccentricity Inclination MeanMotion>;
sink records-summary($pandora-elements-range.values.map({ $_.grep(*.key ∈ @field-names) })».Hash, :@field-names)

+-------------------------+---------------------------------+------------------------------+---------------------------------+
| Date                    | Eccentricity                    | Inclination                  | MeanMotion                      |
+-------------------------+---------------------------------+------------------------------+---------------------------------+
| 2461371.979988426 => 1  | Min    => 0.0002671066884243086 | Min    => 28.02153894359537  | Min    => 0.006569876091982309  |
| 2461371.973043981 => 1  | 1st-Qu => 0.004417458374083654  | 1st-Qu => 28.02187209826335  | 1st-Qu => 0.006569944859249953  |
| 2461371.966099537 => 1  | Mean   => 0.006085297416284951  | Mean   => 28.022046696111122 | Mean   => 0.0065701880706888205 |
| 2461371.917488426 => 1  | Median => 0.006972646929844603  | Median => 28.02191045015411  | Median => 0.006570138185676686  |
| 2461372.084155092 => 1  | 3rd-Qu => 0.008279319156388914  | 3rd-Qu => 28.022358777757905 | 3rd-Qu => 0.006570